# Setup Data

In [2]:
using Pandas
using LinearAlgebra
using JuMP
using HiGHS
using Distributions
using Random
using DataFrames
using Gurobi

In [165]:

mu_py = Pandas.read_pickle("mu.pkl")
stocks_py = mu_py.pyo.index
mu = Vector{Float64}(undef,0)
for i = 1:length(mu_py)
    push!(mu,mu_py.pyo.iloc[i])
end
push!(mu,0.02)
n = length(mu)
short_amount = 0;

In [4]:
stocks = Vector{String}(undef,0)
for i = 1:length(stocks_py)
    push!(stocks,stocks_py[i])
end
push!(stocks,"Non_risk");

In [5]:
sigma_py = Pandas.read_pickle("sigma.pkl")
sigma = Matrix(DataFrames.DataFrame(sigma_py));

In [6]:
d = MvNormal(mu[1:n-1], sigma)

FullNormal(
dim: 462
μ: [0.23584795079384996, 0.20931364446446066, 0.20889248181352235, 0.16463680366755368, 0.16838619455355888, 0.21383129580474694, 0.2614857014706894, 0.245482215330994, 0.23572278313526962, 0.5850107207203755  …  0.2192546042158509, 0.15368489212936043, 0.3267026878519608, 0.17996134487480164, 0.335826518240482, 0.2112173492494937, 0.3172597321563154, 0.25786828956574814, 0.23345687432853185, 0.14125418584375127]
Σ: [26.59746254803952 14.778096639140626 … 8.246911838378587 2.3127228769209958; 14.778096639140626 42.02249312481375 … 10.036320465776466 2.2790116424591056; … ; 8.246911838378587 10.036320465776466 … 9.597599265901119 1.553822202301307; 2.3127228769209958 2.2790116424591056 … 1.553822202301307 5.532684093731671]
)


In [7]:
function expectedshortfall_fast(p:: Vector, d:: Distribution, M:: Int = 2000000)
    Random.seed!(1234)
    X = rand(d, M)   
    X2 = vcat(X, fill(mu[n], 1, size(X, 2)))

    ret = p' * X2     

    losses = ret[ret .< 0]

    loss_prob = length(losses) / M
    es = mean(losses)

    return loss_prob, es
end

expectedshortfall_fast (generic function with 2 methods)

# Uniform Portfolio

In [125]:
p = ones(n)./n


463-element Vector{Float64}:
 0.0021598272138228943
 0.0021598272138228943
 0.0021598272138228943
 0.0021598272138228943
 0.0021598272138228943
 0.0021598272138228943
 0.0021598272138228943
 0.0021598272138228943
 0.0021598272138228943
 0.0021598272138228943
 0.0021598272138228943
 0.0021598272138228943
 0.0021598272138228943
 ⋮
 0.0021598272138228943
 0.0021598272138228943
 0.0021598272138228943
 0.0021598272138228943
 0.0021598272138228943
 0.0021598272138228943
 0.0021598272138228943
 0.0021598272138228943
 0.0021598272138228943
 0.0021598272138228943
 0.0021598272138228943
 0.0021598272138228943

In [129]:
df = DataFrames.DataFrame(Stock=stocks, Position=p)
df.AbsolutePosition = abs.(df.Position)
df.Type = [pos >= 0 ? "Long" : "Short" for pos in df.Position]

# Sort by absolute position and take top 10
top_df = sort(df, :AbsolutePosition, rev=true)[1:min(15, nrow(df)), [:Stock, :Position, :Type]]

println("Diversification Portfolio top 15 Stocks by Absolute Position Size:")
show(top_df, allrows=true)

Diversification Portfolio top 15 Stocks by Absolute Position Size:
15×3 DataFrame
 Row │ Stock   Position    Type   
     │ String  Float64     String 
─────┼────────────────────────────
   1 │ APTV    0.00215983  Long
   2 │ DVN     0.00215983  Long
   3 │ HSY     0.00215983  Long
   4 │ CAG     0.00215983  Long
   5 │ HST     0.00215983  Long
   6 │ LUV     0.00215983  Long
   7 │ MMC     0.00215983  Long
   8 │ BA      0.00215983  Long
   9 │ VRSK    0.00215983  Long
  10 │ TSLA    0.00215983  Long
  11 │ YUM     0.00215983  Long
  12 │ DE      0.00215983  Long
  13 │ CHD     0.00215983  Long
  14 │ PHM     0.00215983  Long
  15 │ XOM     0.00215983  Long

In [10]:
er = sum(p[i]*mu[i] for i = 1:n)
println("Expected return: ", er)
ploss, eshortfall = expectedshortfall_fast(value.(p), d)
sleep(0.02)
println("Loss probability: ", ploss)
println("Expected shortfall: ", eshortfall)
println("Variance: ", value.(p)[1:n-1]'*sigma*value.(p)[1:n-1])

Expected return: 0.2328122589140964
Loss probability: 0.458253
Expected shortfall: -1.6682528134548156
Variance: 4.803074590438797


# Maximizing Return Portfolio

In [12]:
m = Model(Gurobi.Optimizer)

@variable(m, p[1:n] >= short_amount)
@constraint(m, sum(p[i] for i = 1:n) <= 1)

@objective(m, Max, sum(p[i]*mu[i] for i = 1:n));

Set parameter Username
Set parameter LicenseID to value 2742274
Academic license - for non-commercial use only - expires 2026-11-20


In [13]:
optimize!(m)

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: AMD Ryzen 7 7700X 8-Core Processor, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 1 rows, 463 columns and 463 nonzeros
Model fingerprint: 0x9595a54c
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e-02, 6e-01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 1 rows and 463 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.9579551e-01   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  5.957955116e-01

User-callback calls 37, time in user-callback 0.00 sec


In [14]:
df = DataFrames.DataFrame(Stock=stocks, Position=value.(p))
df.AbsolutePosition = abs.(df.Position)
df.Type = [pos >= 0 ? "Long" : "Short" for pos in df.Position]


top_df = sort(df, :AbsolutePosition, rev=true)[1:min(15, nrow(df)), [:Stock, :Position, :Type]]

println("MaxReturn portfolio top 15 Stocks by Absolute Position Size:")
show(top_df, allrows=true)

MaxReturn portfolio top 15 Stocks by Absolute Position Size:
15×3 DataFrame
 Row │ Stock   Position  Type   
     │ String  Float64   String 
─────┼──────────────────────────
   1 │ ENPH         1.0  Long
   2 │ APTV         0.0  Long
   3 │ DVN          0.0  Long
   4 │ HSY          0.0  Long
   5 │ CAG          0.0  Long
   6 │ HST          0.0  Long
   7 │ LUV          0.0  Long
   8 │ MMC          0.0  Long
   9 │ BA           0.0  Long
  10 │ VRSK         0.0  Long
  11 │ TSLA         0.0  Long
  12 │ YUM          0.0  Long
  13 │ DE           0.0  Long
  14 │ CHD          0.0  Long
  15 │ PHM          0.0  Long

In [15]:
println("Expected return: ", objective_value(m))
ploss, eshortfall = expectedshortfall_fast(value.(p), d)
sleep(0.02)
println("Loss probability: ", ploss)
println("Expected shortfall: ", eshortfall)
println("Variance: ", value.(p)[1:n-1]'*sigma*value.(p)[1:n-1])

Expected return: 0.5957955115760344
Loss probability: 0.4782215
Expected shortfall: -8.462982016430146
Variance: 117.99735282302423


# Minimizing Variance Portfolio

In [167]:
mu_bar = er
m = Model(Gurobi.Optimizer)

@variable(m, p[1:n] >= short_amount)
@constraint(m, sum(p[i] for i = 1:n) <= 1)
@constraint(m, sum(p[i]*mu[i] for i = 1:n) >= mu_bar)

@objective(m, Min, p[1:n-1]'*sigma*p[1:n-1])

println(m)

Set parameter Username
Set parameter LicenseID to value 2742274
Academic license - for non-commercial use only - expires 2026-11-20
Min 

Excessive output truncated after 3615382 bytes.

In [169]:
optimize!(m)

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: AMD Ryzen 7 7700X 8-Core Processor, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 2 rows, 463 columns and 926 nonzeros
Model fingerprint: 0x4a89a64f
Model has 106953 quadratic objective terms
Coefficient statistics:
  Matrix range     [2e-02, 1e+00]
  Objective range  [0e+00, 0e+00]
  QObjective range [6e-03, 2e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e-01, 1e+00]
Presolve time: 0.01s
Presolved: 2 rows, 463 columns, 926 nonzeros
Presolved model has 106953 quadratic objective terms
Ordering time: 0.00s

Barrier statistics:
 Free vars  : 461
 AA' NZ     : 1.070e+05
 Factor NZ  : 1.074e+05 (roughly 1 MB of memory)
 Factor Ops : 3.319e+07 (less than 1 second per iteration)
 Threads    : 8

                  Objective                Residual
Iter       Primal          Dual         Primal

In [171]:
df = DataFrames.DataFrame(Stock=stocks, Position=value.(p))
df.AbsolutePosition = abs.(df.Position)
df.Type = [pos >= 0 ? "Long" : "Short" for pos in df.Position]

# Sort by absolute position and take top 10
top_df = sort(df, :AbsolutePosition, rev=true)[1:min(15, nrow(df)), [:Stock, :Position, :Type]]

println("MinVar Portfolio top 15 Stocks by Absolute Position Size:")
show(top_df, allrows=true)

MinVar Portfolio top 15 Stocks by Absolute Position Size:
15×3 DataFrame
 Row │ Stock     Position   Type   
     │ String    Float64    String 
─────┼─────────────────────────────
   1 │ Non_risk  0.25264    Long
   2 │ LLY       0.141464   Long
   3 │ PGR       0.0775017  Long
   4 │ KDP       0.0689143  Long
   5 │ TMUS      0.0390313  Long
   6 │ DPZ       0.0384137  Long
   7 │ KR        0.0354965  Long
   8 │ TTWO      0.0336268  Long
   9 │ WM        0.0332969  Long
  10 │ NVDA      0.0327192  Long
  11 │ NOC       0.0295729  Long
  12 │ ODFL      0.0254887  Long
  13 │ ED        0.0194242  Long
  14 │ ORLY      0.0190591  Long
  15 │ WST       0.0188959  Long

In [173]:
println("Expected return: ", sum(value.(p)[i]*mu[i] for i = 1:n))
ploss, eshortfall = expectedshortfall_fast(value.(p), d)
sleep(0.02)
target_lp = ploss
println("Loss probability: ", ploss)
println("Expected shortfall: ", eshortfall)
println("Variance: ", value.(p)[1:n-1]'*sigma*value.(p)[1:n-1])

Expected return: 0.2328122589146225
Loss probability: 0.425473
Expected shortfall: -0.9041005541959475
Variance: 1.5193125909069476


# Minimizing Loss Probability Portfolio

In [65]:
m = Model(Gurobi.Optimizer)
maxloss = 0
alpha = 1-target_lp
z = 1/quantile(Normal(0,1), alpha)
@variable(m, p[1:n] >= short_amount)
@constraint(m, sum(p[i] for i = 1:n) <= 1)
@constraint(m, [z*(-maxloss+sum(mu[i]*p[i] for i = 1:n)); (sigma^0.5)*p[1:n-1]] in SecondOrderCone())
@objective(m, Max, sum(p[i]*mu[i] for i = 1:n))

Set parameter Username
Set parameter LicenseID to value 2742274
Academic license - for non-commercial use only - expires 2026-11-20


0.23584795079384996 p[1] + 0.20931364446446066 p[2] + 0.20889248181352235 p[3] + 0.16463680366755368 p[4] + 0.16838619455355888 p[5] + 0.21383129580474694 p[6] + 0.2614857014706894 p[7] + 0.245482215330994 p[8] + 0.23572278313526962 p[9] + 0.5850107207203755 p[10] + 0.21092161929074382 p[11] + 0.25689075405561956 p[12] + 0.2202142097211799 p[13] + 0.27913414884789245 p[14] + 0.17142493189533003 p[15] + 0.293696034414582 p[16] + 0.23428327619470835 p[17] + 0.23521780385820618 p[18] + 0.2136380753777793 p[19] + 0.2598370542766638 p[20] + 0.22679356610204393 p[21] + 0.2196741805218378 p[22] + 0.2001348315625897 p[23] + 0.2422610439091028 p[24] + 0.20334051319016452 p[25] + 0.15510909533286732 p[26] + 0.19646083634877798 p[27] + 0.25546860718411946 p[28] + 0.268060187177203 p[29] + 0.27908001231379204 p[30] + 0.18620286630050142 p[31] + 0.26434000634954774 p[32] + 0.15734276819465254 p[33] + 0.2948895600090556 p[34] + 0.20442280425568693 p[35] + 0.17504723246368653 p[36] + 0.28164469825144

In [67]:
optimize!(m)

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: AMD Ryzen 7 7700X 8-Core Processor, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 464 rows, 926 columns and 214833 nonzeros
Model fingerprint: 0x16606236
Model has 1 quadratic constraint
Coefficient statistics:
  Matrix range     [1e-06, 1e+01]
  QMatrix range    [1e+00, 1e+00]
  Objective range  [2e-02, 6e-01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve time: 0.02s
Presolved: 464 rows, 926 columns, 214833 nonzeros
Presolved model has 1 second-order cone constraint
Ordering time: 0.00s

Barrier statistics:
 AA' NZ     : 1.074e+05
 Factor NZ  : 1.079e+05 (roughly 1 MB of memory)
 Factor Ops : 3.341e+07 (less than 1 second per iteration)
 Threads    : 8

                  Objective                Residual
Iter       Primal          Dual         Primal    Dual     Compl  

In [69]:
df = DataFrames.DataFrame(Stock=stocks, Position=value.(p))
df.AbsolutePosition = abs.(df.Position)
df.Type = [pos >= 0 ? "Long" : "Short" for pos in df.Position]

# Sort by absolute position and take top 10
top_df = sort(df, :AbsolutePosition, rev=true)[1:min(15, nrow(df)), [:Stock, :Position, :Type]]

println("MinRisk Portfolio top 15 Stocks by Absolute Position Size:")
show(top_df, allrows=true)

MinRisk Portfolio top 15 Stocks by Absolute Position Size:
15×3 DataFrame
 Row │ Stock     Position   Type   
     │ String    Float64    String 
─────┼─────────────────────────────
   1 │ Non_risk  0.205366   Long
   2 │ LLY       0.150446   Long
   3 │ PGR       0.0824321  Long
   4 │ KDP       0.0732893  Long
   5 │ TMUS      0.0415008  Long
   6 │ DPZ       0.0408435  Long
   7 │ KR        0.037755   Long
   8 │ TTWO      0.0357564  Long
   9 │ WM        0.0354278  Long
  10 │ NVDA      0.0348072  Long
  11 │ NOC       0.031453   Long
  12 │ ODFL      0.0270974  Long
  13 │ ED        0.0206182  Long
  14 │ ORLY      0.0202911  Long
  15 │ WST       0.0200844  Long

In [71]:
er_lp = sum(value.(p)[i]*mu[i] for i = 1:n)
println("Expected return: ", er_lp)
ploss, eshortfall = expectedshortfall_fast(value.(p), d)
sleep(0.02)
println("Loss probability: ", ploss)
println("Expected shortfall: ", eshortfall)
println("Variance: ", value.(p)[1:n-1]'*sigma*value.(p)[1:n-1])

Expected return: 0.24629656407029377
Loss probability: 0.4258305
Expected shortfall: -0.9618467183420845
Variance: 1.7179471815895253


# Maximizing Utility Portfolio

In [27]:
gamma = 0.15
m = Model(Gurobi.Optimizer)

@variable(m, p[1:n] >= short_amount)
@constraint(m, sum(p[i] for i = 1:n) <= 1)

@objective(m, Max, sum(p[i]*mu[i] for i = 1:n) - gamma/2 *( p[1:n-1]'*sigma*p[1:n-1]))

println(m)


Set parameter Username
Set parameter LicenseID to value 2742274
Academic license - for non-commercial use only - expires 2026-11-20
Max 

Excessive output truncated after 3707601 bytes.

In [28]:
optimize!(m)

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: AMD Ryzen 7 7700X 8-Core Processor, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 1 rows, 463 columns and 463 nonzeros
Model fingerprint: 0x74a88cc8
Model has 106953 quadratic objective terms
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e-02, 6e-01]
  QObjective range [4e-04, 2e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve time: 0.01s
Presolved: 1 rows, 463 columns, 463 nonzeros
Presolved model has 106953 quadratic objective terms
Ordering time: 0.00s

Barrier statistics:
 Free vars  : 461
 AA' NZ     : 1.065e+05
 Factor NZ  : 1.070e+05 (roughly 1 MB of memory)
 Factor Ops : 3.298e+07 (less than 1 second per iteration)
 Threads    : 8

                  Objective                Residual
Iter       Primal          Dual         Primal

In [29]:
df = DataFrames.DataFrame(Stock=stocks, Position=value.(p))
df.AbsolutePosition = abs.(df.Position)
df.Type = [pos >= 0 ? "Long" : "Short" for pos in df.Position]

# Sort by absolute position and take top 10
top_df = sort(df, :AbsolutePosition, rev=true)[1:min(15, nrow(df)), [:Stock, :Position, :Type]]

println("MaxUtil Portfolio top 15 Stocks by Absolute Position Size:")
show(top_df, allrows=true)

MaxUtil Portfolio top 15 Stocks by Absolute Position Size:
15×3 DataFrame
 Row │ Stock     Position   Type   
     │ String    Float64    String 
─────┼─────────────────────────────
   1 │ Non_risk  0.302108   Long
   2 │ LLY       0.1321     Long
   3 │ PGR       0.0723711  Long
   4 │ KDP       0.064354   Long
   5 │ TMUS      0.0364466  Long
   6 │ DPZ       0.0358712  Long
   7 │ KR        0.0331472  Long
   8 │ TTWO      0.0314006  Long
   9 │ WM        0.0310942  Long
  10 │ NVDA      0.0305534  Long
  11 │ NOC       0.0276158  Long
  12 │ ODFL      0.0238005  Long
  13 │ ED        0.0181384  Long
  14 │ ORLY      0.0177973  Long
  15 │ WST       0.0176423  Long

In [30]:
println("Expected return: ", sum(value.(p)[i]*mu[i] for i = 1:n))
ploss, eshortfall = expectedshortfall_fast(value.(p), d)
sleep(0.02)
println("Loss probability: ", ploss)
println("Expected shortfall: ", eshortfall)
println("Variance: ", value.(p)[1:n-1]'*sigma*value.(p)[1:n-1])

Expected return: 0.2187260979991341
Loss probability: 0.425025
Expected shortfall: -0.8438230721495293
Variance: 1.324840719543774



# Enforcing Diversification Portfolio

In [115]:
r = er  # Required return
K = 35  # Minimal number of stocks
u = 0.1  # Maximal position size
l = 0.01  # Minimal position size

m = Model(Gurobi.Optimizer)
@variable(m, p[1:n] >= short_amount)
@variable(m, b[1:n], Bin)
@constraint(m, sum(p[i] for i = 1:n) <= 1)
@constraint(m, sum(p[i]*mu[i] for i = 1:n) >= r)
for i = 1:n
    @constraint(m, 0 <= p[i] <= u)
    @constraint(m, p[i] <= b[i])
    @constraint(m, p[i] >= l*b[i] )
end
@constraint(m, sum(b[i] for i = 1:n) >= K)

@objective(m, Min, p[1:n-1]'*sigma*p[1:n-1])
println(m)

Set parameter Username
Set parameter LicenseID to value 2742274
Academic license - for non-commercial use only - expires 2026-11-20
Min 

Excessive output truncated after 3615382 bytes.

In [117]:
optimize!(m)

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: AMD Ryzen 7 7700X 8-Core Processor, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 1392 rows, 1389 columns and 4167 nonzeros
Model fingerprint: 0x3e3bb0d1
Model has 106953 quadratic objective terms
Variable types: 926 continuous, 463 integer (463 binary)
Coefficient statistics:
  Matrix range     [1e-02, 1e+00]
  Objective range  [0e+00, 0e+00]
  QObjective range [6e-03, 2e+02]
  Bounds range     [1e-01, 1e-01]
  RHS range        [2e-01, 4e+01]
Found heuristic solution: objective 4.3492630
Presolve removed 463 rows and 463 columns
Presolve time: 0.03s
Presolved: 929 rows, 926 columns, 3241 nonzeros
Presolved model has 106953 quadratic objective terms
Variable types: 463 continuous, 463 integer (463 binary)

Root relaxation: objective 1.573550e+00, 1383 iterations, 0.01 seconds (0.03 work units)

    No

In [119]:
df = DataFrames.DataFrame(Stock=stocks, Position=value.(p))
df.AbsolutePosition = abs.(df.Position)
df.Type = [pos >= 0 ? "Long" : "Short" for pos in df.Position]

# Sort by absolute position and take top 10
top_df = sort(df, :AbsolutePosition, rev=true)[1:min(36, nrow(df)), [:Stock, :Position, :Type]]

println("Diversification Portfolio top 36 Stocks by Absolute Position Size:")
show(top_df, allrows=true)

Diversification Portfolio top 36 Stocks by Absolute Position Size:
36×3 DataFrame
 Row │ Stock     Position   Type   
     │ String    Float64    String 
─────┼─────────────────────────────
   1 │ LLY       0.1        Long
   2 │ Non_risk  0.1        Long
   3 │ PGR       0.0795067  Long
   4 │ KDP       0.0643706  Long
   5 │ DPZ       0.0404307  Long
   6 │ TMUS      0.0386377  Long
   7 │ KR        0.0366148  Long
   8 │ TTWO      0.0326628  Long
   9 │ NOC       0.0300306  Long
  10 │ NVDA      0.0270198  Long
  11 │ ODFL      0.0233246  Long
  12 │ WM        0.0227231  Long
  13 │ WST       0.0226973  Long
  14 │ CLX       0.0213325  Long
  15 │ MSFT      0.0178464  Long
  16 │ AVGO      0.0151276  Long
  17 │ HRL       0.0151076  Long
  18 │ AZO       0.0148885  Long
  19 │ ED        0.013348   Long
  20 │ WMT       0.0129721  Long
  21 │ ORLY      0.012306   Long
  22 │ MKTX      0.0112596  Long
  23 │ COST      0.01       Long
  24 │ BR        0.01       Long
  25 │ NFLX      0

In [121]:
println("Expected return: ", sum(value.(p)[i]*mu[i] for i = 1:n))
ploss, eshortfall = expectedshortfall_fast(value.(p), d)
sleep(0.02)
println("Loss probability: ", ploss)
println("Expected shortfall: ", eshortfall)
println("Variance: ", value.(p)[1:n-1]'*sigma*value.(p)[1:n-1])

Expected return: 0.23281225891409638
Loss probability: 0.426822
Expected shortfall: -0.9217981649353686
Variance: 1.5746040856327332
